In [63]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver 
# saves all state intermediate as well as final values in RAM and found in cocept of Persistence in Langgraph

In [64]:
load_dotenv()

llm=ChatGroq(model='llama-3.1-8b-instant')

In [65]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explaination:str

In [66]:
def generate_joke(state:JokeState):
    prompt=f'generate a joke for me on topic {state["topic"]}'
    response=llm.invoke(prompt).content
    return {'joke':response}
def generate_explaination(state:JokeState):
    prompt=f'write an explanation for the joke {state["joke"]}'
    response=llm.invoke(prompt).content
    return {'explaination':response}


In [67]:
graph=StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explaination',generate_explaination)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)

checkpointer=InMemorySaver()

workflow=graph.compile(checkpointer=checkpointer)

In [68]:
config1={'configurable':{'thread_id':'1'}}
workflow.invoke({'topic':'pizza'},config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.',
 'explaination': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty" is a play on words. In this joke, "crusty" has a double meaning. \n\nOn one hand, a crusty pizza typically refers to the crispy, golden-brown crust of a pizza. This is a common description used to describe the texture of a well-cooked pizza.\n\nOn the other hand, "crusty" can also be used to describe someone or something that is feeling irritable, grumpy, or a little rough around the edges, much like a crusty old person. \n\nThe joke relies on this wordplay to create a pun, where the pizza\'s crust is used as a metaphor for its emotional state. This wordplay creates a clever and humorous connection between the setup and the punchline, making it a lighthearted and amusing joke.'}

In [69]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.', 'explaination': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty" is a play on words. In this joke, "crusty" has a double meaning. \n\nOn one hand, a crusty pizza typically refers to the crispy, golden-brown crust of a pizza. This is a common description used to describe the texture of a well-cooked pizza.\n\nOn the other hand, "crusty" can also be used to describe someone or something that is feeling irritable, grumpy, or a little rough around the edges, much like a crusty old person. \n\nThe joke relies on this wordplay to create a pun, where the pizza\'s crust is used as a metaphor for its emotional state. This wordplay creates a clever and humorous connection between the setup and the punchline, making it a lighthearted and amusing joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'che

In [70]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.', 'explaination': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty" is a play on words. In this joke, "crusty" has a double meaning. \n\nOn one hand, a crusty pizza typically refers to the crispy, golden-brown crust of a pizza. This is a common description used to describe the texture of a well-cooked pizza.\n\nOn the other hand, "crusty" can also be used to describe someone or something that is feeling irritable, grumpy, or a little rough around the edges, much like a crusty old person. \n\nThe joke relies on this wordplay to create a pun, where the pizza\'s crust is used as a metaphor for its emotional state. This wordplay creates a clever and humorous connection between the setup and the punchline, making it a lighthearted and amusing joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'ch

In [71]:
# Time Travel


In [74]:
workflow.get_state({'configurable':{'thread_id':'1','checkpoint_id':'1f160143-724c-634f-8001-ccd008cd4405'}})

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.'}, next=('generate_explaination',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f160143-724c-634f-8001-ccd008cd4405'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-06-04T12:52:16.596462+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f160143-6c31-6952-8000-998d3cf0f1da'}}, tasks=(PregelTask(id='213ee8eb-9eaf-7eb2-9046-7d6321dd1500', name='generate_explaination', path=('__pregel_pull', 'generate_explaination'), error=None, interrupts=(), state=None, result={'explaination': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty" is a play on words. In this joke, "crusty" has a double meaning. \n\nOn one hand, a crusty pizza typically refers to the crispy, golden-brown crust of a pizza. This is a common description used to describe the t

In [75]:
workflow.invoke(None,{'configurable':{'thread_id':'1','checkpoint_id':'1f160143-724c-634f-8001-ccd008cd4405'}})

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.',
 'explaination': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty" is a play on words. \n\nIn this joke, "crusty" has a double meaning. On the one hand, pizza crust is a literal part of a pizza. However, "feeling a little crusty" is also a common idiomatic expression that means to feel a bit grumpy, irritable, or unwell. \n\nThe humor comes from the unexpected twist on the phrase\'s usual meaning, which is applied to a pizza in a creative and clever way. The joke requires a moment of mental processing to connect the word "crusty" to both the pizza\'s physical characteristic and the idiomatic expression, which creates the comedic effect.'}

In [76]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was feeling a little crusty.', 'explaination': 'The joke "Why did the pizza go to the doctor? Because it was feeling a little crusty" is a play on words. \n\nIn this joke, "crusty" has a double meaning. On the one hand, pizza crust is a literal part of a pizza. However, "feeling a little crusty" is also a common idiomatic expression that means to feel a bit grumpy, irritable, or unwell. \n\nThe humor comes from the unexpected twist on the phrase\'s usual meaning, which is applied to a pizza in a creative and clever way. The joke requires a moment of mental processing to connect the word "crusty" to both the pizza\'s physical characteristic and the idiomatic expression, which creates the comedic effect.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f160144-e1f6-6bd9-8003-a7e0768f834d'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}

In [77]:
# Updating state

In [79]:
workflow.update_state({'configurable':{'thread_id':'1','checkpoint_id':'1f160143-6c31-6952-8000-998d3cf0f1da','checkpoint_ns':''}},{'topic':'sports'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f160157-6be4-6686-8001-437b7fe47dbb'}}

In [81]:
workflow.invoke(None,{'configurable':{'thread_id':'1','checkpoint_id':'1f160157-6be4-6686-8001-437b7fe47dbb'}})

{'topic': 'sports',
 'joke': 'Why did the golfer wear two pairs of pants? \n\nIn case he got a hole in one.',
 'explaination': 'The joke "Why did the golfer wear two pairs of pants? In case he got a hole in one" is a play on words. \n\nIt starts with a common golfing term, "hole in one," which refers to when a golfer hits the ball directly into the hole with one stroke. However, the phrase has a double meaning here. In everyday language, "hole in one" can also refer to a hole in clothing, like a pair of pants.\n\nThe joke relies on this dual meaning to make a pun. The golfer is wearing two pairs of pants to protect themselves from getting a hole (in their pants) if they make a hole in one (a golfing term) during the game. It\'s a clever and humorous use of wordplay, making it a lighthearted and amusing joke.'}

In [83]:
list(workflow.get_state_history({'configurable':{'thread_id':'1'}}))

[StateSnapshot(values={'topic': 'sports', 'joke': 'Why did the golfer wear two pairs of pants? \n\nIn case he got a hole in one.', 'explaination': 'The joke "Why did the golfer wear two pairs of pants? In case he got a hole in one" is a play on words. \n\nIt starts with a common golfing term, "hole in one," which refers to when a golfer hits the ball directly into the hole with one stroke. However, the phrase has a double meaning here. In everyday language, "hole in one" can also refer to a hole in clothing, like a pair of pants.\n\nThe joke relies on this dual meaning to make a pun. The golfer is wearing two pairs of pants to protect themselves from getting a hole (in their pants) if they make a hole in one (a golfing term) during the game. It\'s a clever and humorous use of wordplay, making it a lighthearted and amusing joke.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f160163-5f0e-66ea-8003-c133950cd4dc'}}, metadata={'source': 'loop